# Notebook 47R2 - Advanced Trade-off Analysis + Contact Sheets


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns, cv2
from scipy.spatial.distance import mahalanobis

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
RUN=max([p for p in (ROOT/'observed_print_classifier_v8_6_notebook46').iterdir() if p.is_dir()], key=lambda p:p.stat().st_mtime)
TABLES=RUN/'tables'
OUT=RUN/'notebook47R2_tradeoff_analysis'
FIGS=OUT/'figures'
CONTACT=OUT/'contact_sheets'
for p in [OUT,FIGS,CONTACT]: p.mkdir(parents=True,exist_ok=True)
whole=pd.read_csv(TABLES/'whole_page_candidate_metrics.csv')
inventory=pd.read_csv(TABLES/'whole_page_candidate_inventory.csv')
print(RUN)


In [ ]:
plt.figure(figsize=(10,8))
for r,g in whole.groupby('restorer'): plt.scatter(g.false_inserted_ink,g.missing_ink,label=r,alpha=.5)
plt.legend(); plt.xlabel('False Inserted Ink'); plt.ylabel('Missing Ink'); plt.grid()
plt.savefig(FIGS/'01_page_tradeoff_scatter.png',dpi=300,bbox_inches='tight'); plt.show(); plt.close()


In [ ]:
g=sns.FacetGrid(whole,col='restorer',col_wrap=2,hue='source_group',height=5)
g.map_dataframe(sns.scatterplot,x='false_inserted_ink',y='missing_ink')
g.add_legend(); g.savefig(FIGS/'02_source_group_facet.png'); plt.close('all')


In [ ]:
g=sns.lmplot(data=whole,x='false_inserted_ink',y='missing_ink',hue='source_group',col='restorer',fit_reg=False,height=4)
g.savefig(FIGS/'03_source_group_lmplot.png'); plt.close('all')


In [ ]:
summary=whole.groupby('restorer',as_index=False).agg(mean_false=('false_inserted_ink','mean'),mean_missing=('missing_ink','mean'))
plt.figure(figsize=(8,6))
for _,r in summary.iterrows():
 plt.scatter([r.mean_false],[r.mean_missing],s=200); plt.annotate(r.restorer,(r.mean_false,r.mean_missing))
plt.savefig(FIGS/'04_restorer_mean_tradeoff.png',dpi=300,bbox_inches='tight'); plt.show(); plt.close()


In [ ]:
plt.figure(figsize=(8,6))
pts=summary.sort_values('mean_false')
plt.plot(pts.mean_false,pts.mean_missing,'k--'); plt.scatter(summary.mean_false,summary.mean_missing)
plt.savefig(FIGS/'05_pareto_frontier.png',dpi=300,bbox_inches='tight'); plt.show(); plt.close()


In [ ]:
source_summary=whole.groupby(['source_group','restorer']).agg(false_inserted_ink=('false_inserted_ink','mean'),missing_ink=('missing_ink','mean')).reset_index()
pivot=source_summary.pivot(index='source_group',columns='restorer',values='missing_ink')
plt.figure(figsize=(8,6)); sns.heatmap(pivot,annot=True,cmap='viridis')
plt.savefig(FIGS/'06_variance_heatmap.png',dpi=300,bbox_inches='tight'); plt.show(); plt.close()


In [ ]:
variance=whole.groupby('restorer').agg(false_sd=('false_inserted_ink','std'),missing_sd=('missing_ink','std'))
variance.to_csv(OUT/'variance_summary.csv')


In [ ]:
out=[]
for r,g in whole.groupby('restorer'):
 X=g[['false_inserted_ink','missing_ink']].values
 if len(X)<3: continue
 VI=np.linalg.pinv(np.cov(X.T)); c=X.mean(axis=0)
 d=[mahalanobis(v,c,VI) for v in X]
 t=g.copy(); t['mahalanobis']=d
 out.append(t)
outliers=pd.concat(out)
outliers.to_csv(OUT/'outlier_pages.csv',index=False)


In [ ]:
top=outliers.sort_values('mahalanobis',ascending=False).head(20)
manifest=[]
for _,row in top.iterrows():
 page=str(row['page'])
 subset=inventory[inventory['page'].astype(str)==page]
 imgs={}
 for _,r in subset.iterrows():
  p=str(r['candidate'])
  im=cv2.imread(p,0)
  if im is not None: imgs[str(r['restorer'])]=im
 if len(imgs)==0: continue
 fig,ax=plt.subplots(2,3,figsize=(12,8))
 axes=ax.flatten()
 for i,(name,img) in enumerate(list(imgs.items())[:6]):
  axes[i].imshow(img,cmap='gray',vmin=0,vmax=255); axes[i].set_title(name); axes[i].axis('off')
 for j in range(i+1,6): axes[j].axis('off')
 fn=f'{page}_comparison.png'
 plt.tight_layout(); plt.savefig(CONTACT/fn,dpi=200,bbox_inches='tight'); plt.close()
 manifest.append({'page':page,'sheet':fn})
pd.DataFrame(manifest).to_csv(OUT/'contact_sheet_manifest.csv',index=False)
